### SOME M-T Yves Roland

## Projet Churn — Partie 2 : Deep Learning
### Données textuelles— twcs
---
**Pipeline :** EDA → Prétraitement → Feature Engineering → tf-idf+word2vec+Modèles Deep → Évaluation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

In [ ]:

# Sur Google Colab :
# df = pd.read_csv('/content/drive/MyDrive/twcs.csv')

# En local :
df = pd.read_csv('twcs.csv')


print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
df.head()

(2811774, 7)
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']
tweet_id                     int64
author_id                   object
inbound                       bool
created_at                  object
text                        object
response_tweet_id           object
in_response_to_tweet_id    float64
dtype: object


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [ ]:
print("Messages clients (inbound=True) :", df[df['inbound']==True].shape[0])
print("Messages support (inbound=False) :", df[df['inbound']==False].shape[0])

print("\nExemples de messages clients :")
df[df['inbound']==True]['text'].head(10).to_list()

Messages clients (inbound=True) : 1537843
Messages support (inbound=False) : 1273931

Exemples de messages clients :


['@sprintcare and how do you propose we do that',
 '@sprintcare I have sent several private messages and no one is responding as usual',
 '@sprintcare I did.',
 '@sprintcare is the worst customer service',
 '@sprintcare You gonna magically change your connectivity for me and my whole family ? 🤥 💯',
 '@sprintcare Since I signed up with you....Since day 1',
 '@115714 y’all lie about your “great” connection. 5 bars LTE, still won’t load something. Smh.',
 "@115714 whenever I contact customer support, they tell me I have shortcode enabled on my account, but I have never in the 4 years I've tried https://t.co/0G98RtNxPK",
 '@Ask_Spectrum Would you like me to email you a copy of one since Spectrum is not updating your training?',
 '@Ask_Spectrum I received this from your corporate office would you like a copy?']

In [ ]:
df.duplicated().sum()

np.int64(0)

## Filtrage et échantillonnage:

On filtre inbound=True parce que :

inbound=True = message envoyé par le client → c'est ce qu'on veut analyser
inbound=False = réponse du support → ça ne nous intéresse pas, ce n'est pas le client qui parle


In [ ]:
# Garder uniquement les messages clients (inbound=True)
df_clients = df[df['inbound'] == True][['text']].copy().reset_index(drop=True)

print(f"Messages clients : {df_clients.shape[0]:,}")
print(f"\nExemples :")
print(df_clients['text'].head(5).tolist())

Messages clients : 1,537,843

Exemples :
['@sprintcare and how do you propose we do that', '@sprintcare I have sent several private messages and no one is responding as usual', '@sprintcare I did.', '@sprintcare is the worst customer service', '@sprintcare You gonna magically change your connectivity for me and my whole family ? 🤥 💯']


In [ ]:
# Le dataset complet fait 1.5M messages
# On échantillonne 100 000 messages pour des raisons de contraintes computationnelles
# Les résultats restent représentatifs
df_clients = df_clients.sample(n=100000, random_state=42).reset_index(drop=True)
print(f"Taille : {df_clients.shape[0]:,}")

Taille : 100,000


In [ ]:
df_clients.isnull().sum()

,0
text,0


## Création du label

In [ ]:
from textblob import TextBlob

# TextBlob calcule la polarité de chaque message entre -1 (très négatif) et +1 (très positif)
# On crée un label binaire : 1 = sentiment négatif (risque churn), 0 = positif/neutre
# Lien avec le churn : un sentiment négatif persistant est un signal fort de départ imminent
def get_sentiment(text):
    if not isinstance(text, str):
        return 0
    polarity = TextBlob(text).sentiment.polarity
    return 1 if polarity < 0 else 0

df_clients['sentiment'] = df_clients['text'].apply(get_sentiment)

print("Distribution sentiment :")
print(df_clients['sentiment'].value_counts())
print(f"Taux négatif : {df_clients['sentiment'].mean():.2%}")

Distribution sentiment :
sentiment
0    75100
1    24900
Name: count, dtype: int64
Taux négatif : 24.90%


## Pretraitement du texte

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Lowercase : uniformisation de la casse
    text = text.lower()
    # Suppression des mentions @user : non informatives
    text = re.sub(r'@\w+', '', text)
    # Suppression des URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Suppression des caractères spéciaux et chiffres
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Tokenisation : découpage en mots
    tokens = word_tokenize(text)
    # Suppression des mots vides (the, a, is...) et mots trop courts
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    # Lemmatisation : ramène chaque mot à sa forme de base (running → run)
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

df_clients['text_clean'] = df_clients['text'].apply(clean_text)

print("Avant :", df_clients['text'].iloc[0])
print("Après :", df_clients['text_clean'].iloc[0])

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Avant : @AppleSupport Basically for a chat to be opened from call log, the message app should be opened/running in background. Otherwise, it takes twice.
Après : basically chat opened call log message app opened running background otherwise take twice


In [ ]:
df_clients[['text','text_clean']]

,text,text_clean
0,@AppleSupport Basically for a chat to be opene...,basically chat opened call log message app ope...
1,@AppleSupport iOS 11.02 and Watchos4.0: No ico...,io watchos icon twitter notification restart i...
2,"Dear god not again,@AppleSupport https://t.co/...",dear god
3,@ATVIAssist Hi there! If I buy Call of Duty WW...,buy call duty wwii steam today instant access ...
4,Hi @Safaricom_Care why can't I pay my my Dstv ...,pay dstv text say org unavailable
...,...,...
99995,@Uber_Support hello could you please solve my ...,hello could please solve issue dm
99996,@SouthwestAir Is anyone working in SWABIZ? was...,anyone working swabiz hold hr yesterday hung c...
99997,@ArgosHelpers any chance you’re getting this b...,chance getting back stock price
99998,@BofA_Help No. Your company is greedy. You're ...,company greedy dedicated hit disaster assistin...


## Split + TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

X = df_clients['text_clean']
y = df_clients['sentiment']

# Split stratifié pour conserver la distribution des classes
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF : mesure l'importance de chaque mot dans un message
# par rapport à l'ensemble du corpus
# ngram_range=(1,2) : on considère les mots seuls et les bigrammes (ex: "not happy")
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f"Train : {X_train_tfidf.shape}")
print(f"Test  : {X_test_tfidf.shape}")

Train : (80000, 10000)
Test  : (20000, 10000)


## Baseline TF-IDF + Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Baseline : modèle classique ML sur représentation TF-IDF
# Sert de référence pour évaluer l'apport des modèles Deep Learning
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)

print("=== Baseline TF-IDF + Logistic Regression ===\n")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision : {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred_lr):.4f}")
print(classification_report(y_test, y_pred_lr, target_names=['Positif','Négatif']))

=== Baseline TF-IDF + Logistic Regression ===

Accuracy  : 0.9048
Precision : 0.7912
Recall    : 0.8392
F1-Score  : 0.8145
              precision    recall  f1-score   support

     Positif       0.95      0.93      0.94     15020
     Négatif       0.79      0.84      0.81      4980

    accuracy                           0.90     20000
   macro avg       0.87      0.88      0.88     20000
weighted avg       0.91      0.90      0.91     20000



## Embedding Word2Vec

In [ ]:
!pip install gensim
from gensim.models import Word2Vec

# Préparation des phrases tokenisées pour Word2Vec
sentences = [text.split() for text in X_train]

# Entraînement du modèle Word2Vec
# vector_size : dimension de chaque vecteur mot
# window : contexte de 5 mots autour de chaque mot
# min_count : ignore les mots apparaissant moins de 2 fois
# workers : parallélisation
w2v_model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=42
)

print(f"Vocabulaire Word2Vec : {len(w2v_model.wv):,} mots")
print(f"\nMots similaires à 'bad' :")
print(w2v_model.wv.most_similar('bad', topn=5))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 29.2 MB/s eta 0:00:00
Vocabulaire Word2Vec : 17,042 mots

Mots similaires à 'bad' :
[('terrible', 0.9241399765014648), ('horrible', 0.9209093451499939), ('awful', 0.8785604238510132), ('suck', 0.8735311031341553), ('pathetic', 0.8714022040367126)]


## Vectorisation avec Word2Vec

In [ ]:
# On représente chaque message par la moyenne des vecteurs de ses mots
def text_to_vec(text, model, vector_size=100):
    words = text.split()
    vecs = [model.wv[w] for w in words if w in model.wv]
    if len(vecs) == 0:
        return np.zeros(vector_size)
    return np.mean(vecs, axis=0)

X_train_w2v = np.array([text_to_vec(text, w2v_model) for text in X_train])
X_test_w2v  = np.array([text_to_vec(text, w2v_model) for text in X_test])

print(f"Shape train W2V : {X_train_w2v.shape}")
print(f"Shape test  W2V : {X_test_w2v.shape}")

Shape train W2V : (80000, 100)
Shape test  W2V : (20000, 100)


In [ ]:
print(X_train_w2v[0])


[-0.24814013 -0.27489349 -0.21158142  0.15965053 -0.28506631  0.28166246
  0.20979948  0.22074798 -0.46512493 -0.0635335  -0.27952573 -0.01012867
  0.42015108  0.56826186  0.02392904 -0.16770434  0.16200241 -0.8637172
 -0.17417052  0.03980726  0.2585918   0.41493472  0.24859026  0.45393333
 -0.00865687 -0.07925583 -0.43767619  0.4852241  -0.25767747 -0.35369191
 -0.36375839 -0.45546514  0.15282795 -0.17150186  0.34979653  0.23883481
  0.17889062 -0.50330055  0.18450247  0.19509745 -0.13285823  0.14780819
  0.77351826 -0.38248172  0.01442603 -0.35156021  0.3941378  -0.11863403
  0.67681324 -0.13722579 -0.15300357 -1.22132301 -0.15412624  0.05326987
  0.43807417  0.08646823  0.04866391 -0.11818898  0.24612549  0.3107768
 -0.03208795 -0.12493684 -0.36996731 -0.06414769  0.82719976  0.20615897
  0.39139408 -0.49199858 -0.47847444 -0.50047225  0.13415729 -0.33877397
 -0.19361524 -0.23496315  0.09017414  0.1595666  -0.0813631  -0.14657851
 -0.29070485 -0.01579088  0.33912629 -0.21813755 -0.3

## Baseline Word2Vec + Logistic Regression

In [ ]:
lr_w2v = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_w2v.fit(X_train_w2v, y_train)
y_pred_w2v = lr_w2v.predict(X_test_w2v)

print("=== Word2Vec + Logistic Regression ===\n")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_w2v):.4f}")
print(f"Precision : {precision_score(y_test, y_pred_w2v):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_w2v):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred_w2v):.4f}")
print(classification_report(y_test, y_pred_w2v, target_names=['Positif','Négatif']))

=== Word2Vec + Logistic Regression ===

Accuracy  : 0.6631
Precision : 0.3957
Recall    : 0.6703
F1-Score  : 0.4977
              precision    recall  f1-score   support

     Positif       0.86      0.66      0.75     15020
     Négatif       0.40      0.67      0.50      4980

    accuracy                           0.66     20000
   macro avg       0.63      0.67      0.62     20000
weighted avg       0.74      0.66      0.68     20000



## Tokenisation pour LSTM

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

VOCAB_SIZE = 15000
MAX_LEN = 50

tokenizer_nn = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer_nn.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer_nn.texts_to_sequences(X_train), maxlen=MAX_LEN, padding='post')
X_test_seq  = pad_sequences(tokenizer_nn.texts_to_sequences(X_test),  maxlen=MAX_LEN, padding='post')

print(f"Shape train : {X_train_seq.shape}")
print(f"Shape test  : {X_test_seq.shape}")

Shape train : (80000, 50)
Shape test  : (20000, 50)


## Modèle LSTM

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D, Bidirectional
from tensorflow.keras import backend as K
from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.model_selection import train_test_split

VOCAB_SIZE = 15000
MAX_LEN = 50

K.clear_session()

# Redefine X_train, X_test, y_train, y_test to ensure they are available
X = df_clients['text_clean']
y = df_clients['sentiment']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tokenizer_nn = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer_nn.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer_nn.texts_to_sequences(X_train), maxlen=MAX_LEN, padding='post')
X_test_seq  = pad_sequences(tokenizer_nn.texts_to_sequences(X_test),  maxlen=MAX_LEN, padding='post')
print(f"Shape train : {X_train_seq.shape}")
print(f"Shape test  : {X_test_seq.shape}")

# Création de la matrice d'embeddings Word2Vec
embedding_matrix = np.zeros((VOCAB_SIZE, 100))
for word, idx in tokenizer_nn.word_index.items():
    if idx < VOCAB_SIZE and word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]

print(f"Matrice embeddings : {embedding_matrix.shape}")

model_lstm = Sequential([
    Embedding(VOCAB_SIZE, 100, weights=[embedding_matrix],
              input_length=MAX_LEN, trainable=True),
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(128, dropout=0.3, recurrent_dropout=0.3)),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model_lstm.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)



Passons à son entraînement. Nous utiliserons des poids de classe pour gérer le déséquilibre du jeu de données et `EarlyStopping` pour prévenir le surapprentissage et sauvegarder les meilleurs poids du modèle.

## Entraînement LSTM

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping # Added: Import EarlyStopping

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print(f"Class weights : {class_weight_dict}")

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_lstm = model_lstm.fit(
    X_train_seq, y_train,
    epochs=10,
    batch_size=256,
    validation_split=0.15,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)

Class weights : {0: np.float64(0.6657789613848203), 1: np.float64(2.0080321285140563)}
Epoch 1/10
266/266 ━━━━━━━━━━━━━━━━━━━━ 130s 413ms/step - accuracy: 0.6446 - loss: 0.5903 - val_accuracy: 0.8025 - val_loss: 0.4593
Epoch 2/10
266/266 ━━━━━━━━━━━━━━━━━━━━ 100s 378ms/step - accuracy: 0.8175 - loss: 0.4187 - val_accuracy: 0.8892 - val_loss: 0.3413
Epoch 3/10
266/266 ━━━━━━━━━━━━━━━━━━━━ 139s 367ms/step - accuracy: 0.8945 - loss: 0.3024 - val_accuracy: 0.9067 - val_loss: 0.2763
Epoch 4/10
266/266 ━━━━━━━━━━━━━━━━━━━━ 137s 350ms/step - accuracy: 0.9120 - loss: 0.2573 - val_accuracy: 0.9068 - val_loss: 0.2737
Epoch 5/10
266/266 ━━━━━━━━━━━━━━━━━━━━ 98s 367ms/step - accuracy: 0.9214 - loss: 0.2290 - val_accuracy: 0.9054 - val_loss: 0.2726
Epoch 6/10
266/266 ━━━━━━━━━━━━━━━━━━━━ 95s 358ms/step - accuracy: 0.9270 - loss: 0.2113 - val_accuracy: 0.9034 - val_loss: 0.2846
Epoch 7/10
266/266 ━━━━━━━━━━━━━━━━━━━━ 95s 357ms/step - accuracy: 0.9315 - loss: 0.1967 - val_accuracy: 0.8942 - val_loss:

In [ ]:
model_lstm.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 50, 100)        │     1,500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 50, 100)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 256)            │       234,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,253,029 (20.04 MB)

 Trainable params: 1,751,009 (6.68 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 3,502,020 (13.36 MB)

Après l'entraînement, nous évaluerons les performances de notre modèle LSTM en utilisant diverses métriques comme l'Accuracy, la Précision, le Rappel, le F1-Score et l'ROC-AUC. Nous déterminerons également le seuil de classification optimal.

## Évaluation du Modèle LSTM

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

y_proba_lstm = model_lstm.predict(X_test_seq).flatten()

# Trouver le meilleur seuil
fpr, tpr, thresholds = roc_curve(y_test, y_proba_lstm)
f1_scores = []
for thresh in thresholds:
    y_pred_temp = (y_proba_lstm > thresh).astype(int)
    f1_scores.append(f1_score(y_test, y_pred_temp))

best_thresh = thresholds[np.argmax(f1_scores)]
print(f"Meilleur seuil : {best_thresh:.4f}")

# Évaluation avec le meilleur seuil
y_pred_lstm = (y_proba_lstm > best_thresh).astype(int)

print("\n=== LSTM + Word2Vec Embeddings ===\n")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_lstm):.4f}")
print(f"Precision : {precision_score(y_test, y_pred_lstm):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_lstm):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred_lstm):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_test, y_proba_lstm):.4f}")
print(classification_report(y_test, y_pred_lstm, target_names=['Positif','Négatif']))

625/625 ━━━━━━━━━━━━━━━━━━━━ 42s 66ms/step
Meilleur seuil : 0.6379

=== LSTM + Word2Vec Embeddings ===

Accuracy  : 0.9119
Precision : 0.8158
Recall    : 0.8343
F1-Score  : 0.8250
ROC-AUC   : 0.9397
              precision    recall  f1-score   support

     Positif       0.94      0.94      0.94     15020
     Négatif       0.82      0.83      0.82      4980

    accuracy                           0.91     20000
   macro avg       0.88      0.89      0.88     20000
weighted avg       0.91      0.91      0.91     20000



Passons au modele BERT

## BERT sur 100 000 messages

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import torch

print(f"GPU disponible : {torch.cuda.is_available()}")

MODEL_NAME = "distilbert-base-uncased"
tokenizer_bert = AutoTokenizer.from_pretrained(MODEL_NAME)

# Dataset complet 100 000 messages
df_bert = df_clients[['text_clean', 'sentiment']].copy()
print(f"Taille : {df_bert.shape[0]:,}")
print(f"Distribution :\n{df_bert['sentiment'].value_counts()}")

# Split
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df_bert['text_clean'].tolist(),
    df_bert['sentiment'].tolist(),
    test_size=0.2, random_state=42, stratify=df_bert['sentiment']
)

print(f"\nTrain : {len(train_texts):,} | Test : {len(test_texts):,}")

GPU disponible : True


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Taille : 100,000
Distribution :
sentiment
0    75100
1    24900
Name: count, dtype: int64

Train : 80,000 | Test : 20,000


## Tokenisation BERT

In [ ]:
!pip install transformers datasets -q

from tensorflow.keras.preprocessing.sequence import pad_sequences

def tokenize(texts, batch_size=1000):
    all_encodings = {'input_ids': [], 'attention_mask': []}
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer_bert(
            batch,
            padding='max_length',
            truncation=True,
            max_length=64,
            return_tensors='pt'
        )
        all_encodings['input_ids'].append(enc['input_ids'])
        all_encodings['attention_mask'].append(enc['attention_mask'])
        if i % 10000 == 0:
            print(f"  Tokenisé : {i:,}/{len(texts):,}")
    all_encodings['input_ids']      = torch.cat(all_encodings['input_ids'])
    all_encodings['attention_mask'] = torch.cat(all_encodings['attention_mask'])
    return all_encodings

print("Tokenisation train...")
train_encodings = tokenize(train_texts)
print("Tokenisation test...")
test_encodings  = tokenize(test_texts)
print("Tokenisation terminée")

Tokenisation train...
  Tokenisé : 0/80,000
  Tokenisé : 10,000/80,000
  Tokenisé : 20,000/80,000
  Tokenisé : 30,000/80,000
  Tokenisé : 40,000/80,000
  Tokenisé : 50,000/80,000
  Tokenisé : 60,000/80,000
  Tokenisé : 70,000/80,000
Tokenisation test...
  Tokenisé : 0/20,000
  Tokenisé : 10,000/20,000
Tokenisation terminée


## Dataset PyTorch

In [ ]:
class ChurnDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = ChurnDataset(train_encodings, train_labels)
test_dataset  = ChurnDataset(test_encodings,  test_labels)
print(f"Train : {len(train_dataset):,} | Test : {len(test_dataset):,}")

Train : 80,000 | Test : 20,000


## Entraînement BERT avec GPU

In [ ]:
model_bert = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

training_args = TrainingArguments(
    output_dir='./bert_results',
    num_train_epochs=3,
    per_device_train_batch_size=64,   # GPU → batch size plus grand
    per_device_eval_batch_size=128,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=200,
    learning_rate=2e-5,
    warmup_steps=200,
    weight_decay=0.01,
    fp16=True                         # Précision mixte → entraã nement plus rapide sur GPU
)

trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.233891,0.230138
2,0.208611,0.225404
3,0.169223,0.219933


TrainOutput(global_step=3750, training_loss=0.22945752817789714, metrics={'train_runtime': 396.1756, 'train_samples_per_second': 605.792, 'train_steps_per_second': 9.465, 'total_flos': 3974021959680000.0, 'train_loss': 0.22945752817789714, 'epoch': 3.0})

## Évaluation BERT

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

predictions = trainer.predict(test_dataset)
y_pred_bert = predictions.predictions.argmax(axis=1)
y_test_bert = test_labels

print("=== DistilBERT (100 000 messages) ===\n")
print(f"Accuracy  : {accuracy_score(y_test_bert, y_pred_bert):.4f}")
print(f"Precision : {precision_score(y_test_bert, y_pred_bert):.4f}")
print(f"Recall    : {recall_score(y_test_bert, y_pred_bert):.4f}")
print(f"F1-Score  : {f1_score(y_test_bert, y_pred_bert):.4f}")
print(classification_report(y_test_bert, y_pred_bert, target_names=['Positif','Négatif']))

=== DistilBERT (100 000 messages) ===

Accuracy  : 0.9293
Precision : 0.8719
Recall    : 0.8392
F1-Score  : 0.8552
              precision    recall  f1-score   support

     Positif       0.95      0.96      0.95     15020
     Négatif       0.87      0.84      0.86      4980

    accuracy                           0.93     20000
   macro avg       0.91      0.90      0.90     20000
weighted avg       0.93      0.93      0.93     20000

